In [43]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_len": 1024,
    "embed_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "qkv_bias": False #Query-Key-Value bias in weight matrix
}

In [ ]:
import tiktoken
import torch
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

tokenizer.encode(txt1)
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


### Layer Norm

In [14]:
import torch.nn as nn
torch.manual_seed(123)
batch_example = torch.randn(2,5)
print(batch_example)
layer = nn.Sequential(nn.Linear(5,6), nn.ReLU())
out = layer(batch_example)
print(out)
mean = out.mean(dim=-1, keepdim=True)
print(mean)
var = out.var(dim=-1, keepdim=True)
print(var)

# normalize the layer
out_norm = (out - mean) / torch.sqrt(var)
print(out_norm)
mean = out_norm.mean(dim=-1, keepdim=True)
print(mean)
var = out_norm.var(dim=-1, keepdim=True)
print(var)


tensor([[-0.1115,  0.1204, -0.3696, -0.2404, -1.1969],
        [ 0.2093, -0.9724, -0.7550,  0.3239, -0.1085]])
tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)
tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)
tensor([[ 0.6159,  1.4126, -0.8719,  0.5872, -0.8719, -0.8719],
        [-0.0189,  0.1121, -1.0876,  1.5173,  0.5647, -1.0876]],
       grad_fn=<DivBackward0>)
tensor([[-5.9605e-08],
        [ 1.9868e-08]], grad_fn=<MeanBackward1>)
tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [18]:
class LayerNorm(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(embed_dim))
        self.shift = nn.Parameter(torch.zeros(embed_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True)
        # normalize the layer
        out_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * out_norm + self.shift

In [ ]:
# Compare our implementation with Pytorch for LayerNorm
layer_n_torch = nn.LayerNorm(5)
out_norm = layer_n_torch(batch_example)
print(out_norm)

mean = out_norm.mean(dim=-1, keepdim=True)
print(mean)
var = out_norm.var(dim=-1, keepdim=True)
print(var)

layer_n = LayerNorm(5)
out_norm = layer_n(batch_example)
print(out_norm)

mean = out_norm.mean(dim=-1, keepdim=True)
print(mean)
var = out_norm.var(dim=-1, keepdim=True)
print(var)

tensor([[ 0.5528,  1.0693, -0.0223,  0.2656, -1.8654],
        [ 0.9087, -1.3767, -0.9564,  1.1304,  0.2940]],
       grad_fn=<NativeLayerNormBackward0>)
tensor([[-3.5763e-08],
        [ 2.3842e-08]], grad_fn=<MeanBackward1>)
tensor([[1.2499],
        [1.2500]], grad_fn=<VarBackward0>)
tensor([[ 0.4945,  0.9564, -0.0200,  0.2375, -1.6685],
        [ 0.8127, -1.2313, -0.8554,  1.0110,  0.2630]], grad_fn=<AddBackward0>)
tensor([[-1.4901e-08],
        [ 2.3842e-08]], grad_fn=<MeanBackward1>)
tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


### Feedforward network

In [25]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["embed_dim"], 4 * cfg["embed_dim"]),
            nn.GELU(),
            nn.Linear(4 * cfg["embed_dim"], cfg["embed_dim"])
        )
    def forward(self, x):
        return self.layers(x)

Shortcut connections were proposed for deep networks and computer vision specifically in residual networks to mitigate the challenge of vanishing ingredients. Shortcut connections involve adding the input of a layer to its outputs effectively creating an alternate path that bypasses certain layers.

In [31]:
class ResidualNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]),nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]),nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]),nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]),nn.GELU()),
        ])
    def forward(self, x):
        for layer in self.layers:
            layer_out = layer(x)
            if self.use_shortcut and x.shape == layer_out.shape:
                x = x + layer_out
            else:
                x = layer_out
        return x
    
def print_gradients(model, x):
    out = model(x)
    target = torch.tensor([[0.]])
    loss = nn.MSELoss()
    loss = loss(out, target)
    loss.backward()
    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"{name} has gradient mean = {param.grad.abs().mean().item()}")


In [35]:
layer_sizes = [3,3,3,3,1]
sample_inp = torch.tensor([[-1., 0., 1.]])
model_no_residual = ResidualNetwork(layer_sizes, use_shortcut=False)

print_gradients(model_no_residual, sample_inp)
print('-------')
model_no_residual = ResidualNetwork(layer_sizes, use_shortcut=True)

print_gradients(model_no_residual, sample_inp)

layers.0.0.weight has gradient mean = 1.0909249795076903e-05
layers.1.0.weight has gradient mean = 8.844858712109271e-06
layers.2.0.weight has gradient mean = 6.688045687042177e-05
layers.3.0.weight has gradient mean = 0.00012817724200431257
-------
layers.0.0.weight has gradient mean = 0.0029092817567288876
layers.1.0.weight has gradient mean = 0.004488850012421608
layers.2.0.weight has gradient mean = 0.0047469851560890675
layers.3.0.weight has gradient mean = 0.024451980367302895


## Transformer Block

The shape of the input tensor and output tensor in transformer block is the same. This preservation of shape throughout the transformer block architecture is not incidental, but a crucial aspect of its design. This design enables its effective application across a wide range of sequence for sequence tasks, where each act directly correspond to an input factor maintaining a one to one relationship. However, the output is a context vector that encapsulate information from the entire input sequence via multihead attention. This means that while the physical dimensions of the sequence length and feature size remain unchanged as it passes to the transformer block, the content of each output vector is recoded to integrate contextual information from a across the entire input sequence.

In [44]:
import sys
sys.path.append('.')
from llms_from_scratch_core import MultiHeadAttention

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["embed_dim"],
            d_out=cfg["embed_dim"],
            context_length=cfg["context_len"],
            num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = nn.LayerNorm(cfg["embed_dim"])
        self.norm2 = nn.LayerNorm(cfg["embed_dim"])
    
    def forward(self, x):
        # Attention
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = x + shortcut
        # Feedforward
        shortcut = x
        x = self.norm1(x)
        x = self.ff(x)
        x = x + shortcut
        return x
    
x = torch.rand(2,4, 768)
block = TransformerBlock(GPT_CONFIG_124M)
out = block(x)

print(f"in {x.shape} ... out {out.shape}")
    

attn_scores torch.Size([2, 12, 4, 4])
torch.Size([2, 12, 4, 64])
torch.Size([2, 4, 12, 64])
torch.Size([2, 4, 768])
in torch.Size([2, 4, 768]) ... out torch.Size([2, 4, 768])


## GPT Architecture

In [46]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["embed_dim"])
        self.pos_emb = nn.Embedding(cfg["context_len"], cfg["embed_dim"])
        
        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = nn.LayerNorm(cfg["embed_dim"])
        self.out_head = nn.Linear(cfg["embed_dim"], cfg["vocab_size"],
                                   bias=False)
    
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device = in_idx.device))

        x = tok_embeds + pos_embeds
        x = self.transformer_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

model = GPTModel(GPT_CONFIG_124M)
out = model(batch)
print(f"in {batch.shape} ... out {out.shape}")



attn_scores torch.Size([2, 12, 4, 4])
torch.Size([2, 12, 4, 64])
torch.Size([2, 4, 12, 64])
torch.Size([2, 4, 768])
attn_scores torch.Size([2, 12, 4, 4])
torch.Size([2, 12, 4, 64])
torch.Size([2, 4, 12, 64])
torch.Size([2, 4, 768])
attn_scores torch.Size([2, 12, 4, 4])
torch.Size([2, 12, 4, 64])
torch.Size([2, 4, 12, 64])
torch.Size([2, 4, 768])
attn_scores torch.Size([2, 12, 4, 4])
torch.Size([2, 12, 4, 64])
torch.Size([2, 4, 12, 64])
torch.Size([2, 4, 768])
attn_scores torch.Size([2, 12, 4, 4])
torch.Size([2, 12, 4, 64])
torch.Size([2, 4, 12, 64])
torch.Size([2, 4, 768])
attn_scores torch.Size([2, 12, 4, 4])
torch.Size([2, 12, 4, 64])
torch.Size([2, 4, 12, 64])
torch.Size([2, 4, 768])
attn_scores torch.Size([2, 12, 4, 4])
torch.Size([2, 12, 4, 64])
torch.Size([2, 4, 12, 64])
torch.Size([2, 4, 768])
attn_scores torch.Size([2, 12, 4, 4])
torch.Size([2, 12, 4, 64])
torch.Size([2, 4, 12, 64])
torch.Size([2, 4, 768])
attn_scores torch.Size([2, 12, 4, 4])
torch.Size([2, 12, 4, 64])
torch.S

In [47]:
total_params = sum(p.numel() for p in model.parameters())
print(f"total params {total_params}")

total params 163009536


In [51]:
def generate_text_greedy(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:,-1,:]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [53]:
encoded = tokenizer.encode("Hello, I am")
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print(encoded_tensor)
model.eval()
out = generate_text_greedy(model=model, idx=encoded_tensor, 
                           max_new_tokens=6, context_size=GPT_CONFIG_124M["context_len"])
print(out)
decoded_txt = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_txt)

tensor([[15496,    11,   314,   716]])
attn_scores torch.Size([1, 12, 4, 4])
torch.Size([1, 12, 4, 64])
torch.Size([1, 4, 12, 64])
torch.Size([1, 4, 768])
attn_scores torch.Size([1, 12, 4, 4])
torch.Size([1, 12, 4, 64])
torch.Size([1, 4, 12, 64])
torch.Size([1, 4, 768])
attn_scores torch.Size([1, 12, 4, 4])
torch.Size([1, 12, 4, 64])
torch.Size([1, 4, 12, 64])
torch.Size([1, 4, 768])
attn_scores torch.Size([1, 12, 4, 4])
torch.Size([1, 12, 4, 64])
torch.Size([1, 4, 12, 64])
torch.Size([1, 4, 768])
attn_scores torch.Size([1, 12, 4, 4])
torch.Size([1, 12, 4, 64])
torch.Size([1, 4, 12, 64])
torch.Size([1, 4, 768])
attn_scores torch.Size([1, 12, 4, 4])
torch.Size([1, 12, 4, 64])
torch.Size([1, 4, 12, 64])
torch.Size([1, 4, 768])
attn_scores torch.Size([1, 12, 4, 4])
torch.Size([1, 12, 4, 64])
torch.Size([1, 4, 12, 64])
torch.Size([1, 4, 768])
attn_scores torch.Size([1, 12, 4, 4])
torch.Size([1, 12, 4, 64])
torch.Size([1, 4, 12, 64])
torch.Size([1, 4, 768])
attn_scores torch.Size([1, 12, 4,